# Creating the research panel

In this notebook, I create the research panel required for the project.

I import both Fama-French three-factor and five-factor values. The monthly data is obtained from https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html.

The structure of the notebook is as follows:
- Importing daily prices data and calculating current and forward (target for rolling betas) monthly returns
- Research panel creation
- Importing and processing the Fama-French factors (ff3 and ff5) data
- Calculate excess monthly returns and export

In [1]:
import pandas as pd
import numpy as np
import re
from io import StringIO

## Import daily prices data and calculate current and forward monthly returns

In [2]:
prices = pd.read_parquet("./parquets/daily_prices_stooq.parquet").copy()
prices["date"] = pd.to_datetime(prices["date"])

# basic cleaning
prices = prices.sort_values(["ticker", "date"]).reset_index(drop=True)

# keep only columns needed for return construction
prices = prices[["ticker", "date", "close", "volume", "exchange"]].copy()

# drop invalid rows
prices = prices.dropna(subset=["ticker", "date", "close"])
prices = prices[prices["close"] > 0].copy()
prices.head()

,ticker,date,close,volume,exchange
0,A,1999-11-18,28.5858,6.886678e+07,NYSE
1,A,1999-11-19,26.2311,1.677358e+07,NYSE
2,A,1999-11-22,28.5858,7.242576e+06,NYSE
3,A,1999-11-23,25.9889,6.579458e+06,NYSE
4,A,1999-11-24,26.6745,5.332648e+06,NYSE


In [3]:
prices["month"] = prices["date"].dt.to_period("M")

# create a month end panel for clean processing
month_end_prices = (
    prices
    .sort_values(["ticker", "date"])
    .groupby(["ticker", "month"], as_index=False)
    .last()
)

month_end_prices = month_end_prices.rename(columns={
    "date": "month_end_date",
    "close": "month_end_close",
    "volume": "month_end_volume"
})

month_end_prices.head()

,ticker,month,month_end_date,month_end_close,month_end_volume,exchange
0,A,1999-11,1999-11-30,27.4110,4.745571e+06,NYSE
1,A,1999-12,1999-12-31,50.2260,2.126347e+06,NYSE
2,A,2000-01,2000-01-31,43.0013,1.145527e+06,NYSE
3,A,2000-02,2000-02-29,67.4805,1.054862e+06,NYSE
4,A,2000-03,2000-03-31,67.5622,4.110159e+06,NYSE


In [4]:
month_end_prices = month_end_prices.sort_values(["ticker", "month"]).reset_index(drop=True)

# previous month info
month_end_prices["prev_month_end_close"] = (
    month_end_prices.groupby("ticker")["month_end_close"].shift(1)
)
month_end_prices["prev_month"] = (
    month_end_prices.groupby("ticker")["month"].shift(1)
)

# next month info
month_end_prices["next_month_end_close"] = (
    month_end_prices.groupby("ticker")["month_end_close"].shift(-1)
)
month_end_prices["next_month"] = (
    month_end_prices.groupby("ticker")["month"].shift(-1)
)

# check if months are consecutive
month_end_prices["expected_prev_month"] = month_end_prices["month"] - 1
month_end_prices["expected_next_month"] = month_end_prices["month"] + 1
is_consecutive_prev = month_end_prices["prev_month"] == month_end_prices["expected_prev_month"]
is_consecutive_next = month_end_prices["next_month"] == month_end_prices["expected_next_month"]

# realised current-month return: t-1 -> t
month_end_prices["ret_1m"] = (
    month_end_prices["month_end_close"] / month_end_prices["prev_month_end_close"] - 1
)
month_end_prices.loc[~is_consecutive_prev, "ret_1m"] = np.nan

# forward return: t -> t+1
month_end_prices["ret_fwd_1m"] = (
    month_end_prices["next_month_end_close"] / month_end_prices["month_end_close"] - 1
)
month_end_prices.loc[~is_consecutive_next, "ret_fwd_1m"] = np.nan

# make the panel
month_panel = month_end_prices[
    ["ticker", "month", "month_end_date", "month_end_close", "ret_1m", "ret_fwd_1m"]
].copy()

print(month_panel.head(10))
month_panel["ret_1m"].describe()

  ticker    month month_end_date  month_end_close    ret_1m  ret_fwd_1m
0      A  1999-11     1999-11-30          27.4110       NaN    0.832330
1      A  1999-12     1999-12-31          50.2260  0.832330   -0.143844
2      A  2000-01     2000-01-31          43.0013 -0.143844    0.569267
3      A  2000-02     2000-02-29          67.4805  0.569267    0.001211
4      A  2000-03     2000-03-31          67.5622  0.001211   -0.147834
5      A  2000-04     2000-04-28          57.5742 -0.147834   -0.169275
6      A  2000-05     2000-05-31          47.8283 -0.169275    0.001773
7      A  2000-06     2000-06-30          47.9131  0.001773   -0.447433
8      A  2000-07     2000-07-31          26.4752 -0.447433    0.496914
9      A  2000-08     2000-08-31          39.6311  0.496914   -0.197736


count    990743.000000
mean          0.010723
std           0.385676
min          -0.999648
25%          -0.054363
50%           0.003032
75%           0.058952
max         302.636364
Name: ret_1m, dtype: float64

## Create the research panel

In [5]:
universe = pd.read_parquet("./parquets/monthly_top1500_liquidity.parquet").copy()

# make sure month is the same dtype
if "month" in universe.columns:
    universe["month"] = pd.PeriodIndex(universe["month"], freq="M")

universe.head()

,ticker,date,open,high,low,close,volume,exchange,dollar_volume,median_dollar_volume_60d,month,liquid_rank
0,A,2000-02-29,65.9020,69.4366,65.8183,67.4805,1.054862e+06,NYSE,7.118264e+07,8.717407e+07,2000-02,76.0
1,A,2000-03-31,68.8626,68.8626,58.4711,67.5622,4.110159e+06,NYSE,2.776914e+08,1.057878e+08,2000-03,74.0
2,A,2000-04-28,58.4711,58.5089,56.4810,57.5742,1.959183e+06,NYSE,1.127984e+08,1.396292e+08,2000-04,60.0
3,A,2000-05-31,48.0755,50.7980,47.2613,47.8283,4.853778e+06,NYSE,2.321480e+08,1.886477e+08,2000-05,50.0
4,A,2000-06-30,49.8234,51.6490,47.8553,47.9131,9.116944e+06,NYSE,4.368211e+08,2.456412e+08,2000-06,38.0


In [6]:
# make the research panel
research_panel = universe.merge(
    month_panel,
    on=["ticker", "month"],
    how="left",
    validate="one_to_one"
)

research_panel.head()

,ticker,date,open,high,low,close,volume,exchange,dollar_volume,median_dollar_volume_60d,month,liquid_rank,month_end_date,month_end_close,ret_1m,ret_fwd_1m
0,A,2000-02-29,65.9020,69.4366,65.8183,67.4805,1.054862e+06,NYSE,7.118264e+07,8.717407e+07,2000-02,76.0,2000-02-29,67.4805,0.569267,0.001211
1,A,2000-03-31,68.8626,68.8626,58.4711,67.5622,4.110159e+06,NYSE,2.776914e+08,1.057878e+08,2000-03,74.0,2000-03-31,67.5622,0.001211,-0.147834
2,A,2000-04-28,58.4711,58.5089,56.4810,57.5742,1.959183e+06,NYSE,1.127984e+08,1.396292e+08,2000-04,60.0,2000-04-28,57.5742,-0.147834,-0.169275
3,A,2000-05-31,48.0755,50.7980,47.2613,47.8283,4.853778e+06,NYSE,2.321480e+08,1.886477e+08,2000-05,50.0,2000-05-31,47.8283,-0.169275,0.001773
4,A,2000-06-30,49.8234,51.6490,47.8553,47.9131,9.116944e+06,NYSE,4.368211e+08,2.456412e+08,2000-06,38.0,2000-06-30,47.9131,0.001773,-0.447433


In [7]:
print("Rows in universe:", len(universe))
print("Rows after merge:", len(research_panel))
print("Missing forward returns:", research_panel["ret_fwd_1m"].isna().mean())

research_panel["ret_fwd_1m"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])

Rows in universe: 445308
Rows after merge: 445308
Missing forward returns: 0.0033931570957629327


count    443797.000000
mean          0.010708
std           0.123278
min          -0.957500
1%           -0.294275
5%           -0.161132
50%           0.008968
95%           0.183512
99%           0.345856
max          16.250531
Name: ret_fwd_1m, dtype: float64

In [8]:
research_panel.loc[research_panel["ticker"] == "AAPL",
                   ["ticker", "month", "month_end_date", "month_end_close", "ret_1m", "ret_fwd_1m"]].tail(10)

,ticker,month,month_end_date,month_end_close,ret_1m,ret_fwd_1m
2043,AAPL,2025-06,2025-06-30,204.739,0.021509,0.011698
2044,AAPL,2025-07,2025-07-31,207.134,0.011698,0.119638
2045,AAPL,2025-08,2025-08-29,231.915,0.119638,0.096880
2046,AAPL,2025-09,2025-09-30,254.383,0.096880,0.061816
2047,AAPL,2025-10,2025-10-31,270.108,0.061816,0.032365
2048,AAPL,2025-11,2025-11-28,278.850,0.032365,-0.025067
2049,AAPL,2025-12,2025-12-31,271.860,-0.025067,-0.045538
2050,AAPL,2026-01,2026-01-30,259.480,-0.045538,0.018113
2051,AAPL,2026-02,2026-02-27,264.180,0.018113,-0.016239
2052,AAPL,2026-03,2026-03-09,259.890,-0.016239,NaN


In [9]:
# see monthly summary
monthly_summary = (
    research_panel.groupby("month")["ret_1m"]
    .agg(["count", "mean", "median", "std", "min", "max"])
)

monthly_summary.tail()

,count,mean,median,std,min,max
month,,,,,,
2025-11,1500,0.002168,0.014377,0.141074,-0.642857,1.041979
2025-12,1500,-0.001837,-0.006098,0.106412,-0.678768,1.064073
2026-01,1500,0.038468,0.034851,0.140091,-0.415179,1.427542
2026-02,1500,0.016688,0.019997,0.157190,-0.611332,1.246955
2026-03,1500,-0.026404,-0.030133,0.073611,-0.248270,0.717576


In [10]:
# check for anomalous months
extreme = research_panel.loc[
    research_panel["ret_1m"].abs() > 1.0,
    ["ticker", "month", "month_end_close", "ret_1m", "ret_fwd_1m"]
].sort_values("ret_1m")

print(extreme.tail(20))

       ticker    month  month_end_close     ret_1m  ret_fwd_1m
31908    ARMP  2000-02     1.496250e+06   2.525773   -0.350877
340440   RIGL  2007-12     2.539000e+02   2.546089    0.084285
259527    MGM  2009-04     7.917240e+00   2.596244   -0.109779
320448    PLD  2008-12     2.204160e+01   2.626587   -0.279267
4237     ACCO  2008-12     2.844350e+00   2.750333   -0.443521
182100   GREE  2021-08     1.489100e+03   2.910349   -0.828487
25645      AN  1995-05     5.970000e+00   2.927632   -0.018425
332971     QS  2020-11     4.700000e+01   2.983051    0.796809
256039   MDGL  2022-12     2.902500e+02   3.139923   -0.006891
30285      AR  2020-04     2.980000e+00   3.180109    0.003356
339162   RGTI  2024-12     1.526000e+01   4.003279   -0.136959
153333   FCEL  2020-11     3.060000e+02   4.100000    0.095098
24278    AMRN  2018-09     3.254000e+02   4.148734    0.280270
296518   OCGN  2021-02     1.095000e+01   5.186441   -0.379909
21495     AMC  2021-01     1.325480e+02   5.254713   -0

## Import Fama-French factors and merge onto the research panel

In [11]:
path_ff3 = "./ff data/F-F_Research_Data_Factors.csv"
path_ff5 = "./ff data/F-F_Research_Data_5_Factors_2x3.csv"

# extract the monthly factors rows
def get_monthly_lines(path):
    with open(path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    monthly_lines = []
    for line in lines:
        stripped = line.strip()
        if not stripped:
            continue
        
        # match for YYYYMM
        first_field = stripped.split(",")[0]
        if re.fullmatch(r"\d{6}", first_field):
            monthly_lines.append(line)

    return monthly_lines

monthly_lines_ff3 = get_monthly_lines(path_ff3)
monthly_lines_ff5 = get_monthly_lines(path_ff5)

monthly_csv_ff3 = "date_raw,Mkt-RF,SMB,HML,RF\n" + "".join(monthly_lines_ff3)
monthly_csv_ff5 = "date_raw,Mkt-RF,SMB,HML,RMW,CMA,RF\n" + "".join(monthly_lines_ff5)
ff3 = pd.read_csv(StringIO(monthly_csv_ff3))
ff5 = pd.read_csv(StringIO(monthly_csv_ff5))

ff3 = ff3.rename(columns={
    "Mkt-RF": "mkt_rf",
    "SMB": "smb",
    "HML": "hml",
    "RF": "rf",
})

ff5 = ff5.rename(columns={
    "Mkt-RF": "mkt_rf",
    "SMB": "smb",
    "HML": "hml",
    "RMW": "rmw",
    "CMA": "cma",
    "RF": "rf",
})

ff3["month"] = pd.PeriodIndex(ff3["date_raw"].astype(str), freq="M")
ff5["month"] = pd.PeriodIndex(ff5["date_raw"].astype(str), freq="M")
ff3["date"] = ff3["month"].dt.to_timestamp("M")
ff5["date"] = ff5["month"].dt.to_timestamp("M")

# convert to numeric
factor_cols_ff3 = ["mkt_rf", "smb", "hml", "rf"]
factor_cols_ff5 = ["mkt_rf", "smb", "hml", "rmw", "cma", "rf"]
ff3[factor_cols_ff3] = ff3[factor_cols_ff3].astype(float) / 100.0
ff5[factor_cols_ff5] = ff5[factor_cols_ff5].astype(float) / 100.0

ff3 = ff3[["month", "date", "mkt_rf", "smb", "hml", "rf"]].sort_values("month").reset_index(drop=True)
ff5 = ff5[["month", "date", "mkt_rf", "smb", "hml", "rmw", "cma", "rf"]].sort_values("month").reset_index(drop=True)

# ff3 data is from 1926 while ff5 is from 1963
print(ff3[factor_cols_ff3].describe())
print(ff5[factor_cols_ff5].describe())

            mkt_rf          smb          hml           rf
count  1195.000000  1195.000000  1195.000000  1195.000000
mean      0.006918     0.001660     0.003501     0.002701
std       0.053053     0.031514     0.035546     0.002488
min      -0.287400    -0.174100    -0.138300    -0.000600
25%      -0.019950    -0.016250    -0.014300     0.000300
50%       0.010800     0.000600     0.001400     0.002300
75%       0.036550     0.017350     0.017600     0.004200
max       0.388100     0.359600     0.355200     0.013500
           mkt_rf         smb         hml         rmw         cma          rf
count  751.000000  751.000000  751.000000  751.000000  751.000000  751.000000
mean     0.005949    0.001835    0.002911    0.002644    0.002428    0.003634
std      0.044562    0.030268    0.029720    0.022208    0.020628    0.002613
min     -0.231900   -0.155400   -0.138300   -0.189500   -0.070800    0.000000
25%     -0.019600   -0.015800   -0.014400   -0.008450   -0.010350    0.001500
50%      0

In [12]:
research_panel["month"] = pd.PeriodIndex(research_panel["month"], freq="M")
ff3["month"] = pd.PeriodIndex(ff3["month"], freq="M")
ff5["month"] = pd.PeriodIndex(ff5["month"], freq="M")

panel_ff3 = research_panel.merge(
    ff3,
    on="month",
    how="left",
    validate="many_to_one"
)

panel_ff5 = research_panel.merge(
    ff5,
    on="month",
    how="left",
    validate="many_to_one"
)

## Calculate excess return and export research panel to parquet

In [13]:
panel_ff3["excess_ret_1m"] = panel_ff3["ret_1m"] - panel_ff3["rf"]
panel_ff5["excess_ret_1m"] = panel_ff5["ret_1m"] - panel_ff5["rf"]

In [14]:
# remove unwanted duplicate columns
print([c for c in panel_ff3.columns if "date" in c.lower()])
print([c for c in panel_ff3.columns if "month" in c.lower()])

['date_x', 'month_end_date', 'date_y']
['month', 'month_end_date', 'month_end_close']


In [15]:
keep_cols_ff3 = [
    "ticker",
    "month", "month_end_date", "month_end_close",
    "ret_1m", "ret_fwd_1m", "excess_ret_1m",
    "mkt_rf", "smb", "hml", "rf",
]

keep_cols_ff5 = [
    "ticker",
    "month", "month_end_date", "month_end_close",
    "ret_1m", "ret_fwd_1m", "excess_ret_1m",
    "mkt_rf", "smb", "hml", "rmw", "cma", "rf",
]

# keep only columns that actually exist
keep_cols_ff3 = [c for c in keep_cols_ff3 if c in panel_ff3.columns]
keep_cols_ff5 = [c for c in keep_cols_ff5 if c in panel_ff5.columns]
panel_ff3 = panel_ff3[keep_cols_ff3].copy()
panel_ff5 = panel_ff5[keep_cols_ff5].copy()

panel_ff3 = panel_ff3.sort_values(["ticker", "month"]).reset_index(drop=True)
panel_ff5 = panel_ff5.sort_values(["ticker", "month"]).reset_index(drop=True)

# export
panel_ff3.to_parquet("./parquets/monthly_top1500_and_ff3.parquet")
panel_ff5.to_parquet("./parquets/monthly_top1500_and_ff5.parquet")